In [ ]:
#!/usr/init/env python3
"""Annotate a local video with YOLOv8 cow/cowdung detections.

Setup (Python 3.10+):
    python -m pip install ultralytics opencv-python numpy
"""
from __future__ import annotations

import argparse
from dataclasses import dataclass
import logging
import math
from pathlib import Path
import sys

import cv2
import numpy as np
from ultralytics import YOLO

# ==========================================
# CONFIGURATION (Edit these for your notebook)
# ==========================================
VIDEO_PATH = r'C:\Users\User\Downloads\videoplayback.mp4'  
OUTPUT_PATH = 'annotated_output.mp4'
CONF_THRESH = 0.35
MODEL_PATH = 'yolov8n.pt'  # Change to 'best.pt' if using custom trained weights
ALLOW_MISSING = True       # Set to False if your model has both cow and cowdung classes
CLASS_ALIASES = {'cow': 'cow', 'cowdung': 'cowdung'}
COLORS = {'cow': (0, 200, 0), 'cowdung': (0, 140, 255)}  # OpenCV BGR format


@dataclass(frozen=True)
class Detection:
    label: str
    confidence: float
    xyxy: tuple[int, int, int, int]


def load_model(weights: str, allow_missing: bool = False):
    model = YOLO(weights)
    if model.task != 'detect':
        raise ValueError('Use object-detection weights, not classification/pose weights.')
    names = model.names
    items = names.items() if isinstance(names, dict) else enumerate(names)
    class_map = {}
    for class_id, name in items:
        normalized = str(name).lower().strip().replace('_', '').replace('-', '').replace(' ', '')
        if normalized in CLASS_ALIASES:
            class_map[int(class_id)] = CLASS_ALIASES[normalized]
    missing = set(COLORS) - set(class_map.values())
    if missing:
        message = f'Model lacks target classes: {sorted(missing)}. Use weights trained on both classes.'
        if not allow_missing:
            raise ValueError(message + ' For a partial demonstration, pass allow_missing=True.')
        logging.warning(message)
    if not class_map:
        raise ValueError('Model has no recognized target classes.')
    logging.info('Using target class IDs: %s', class_map)
    return model, class_map


def detect_frame(model, frame: np.ndarray, class_map: dict[int, str],
                 conf: float = CONF_THRESH, imgsz: int = 640,
                 device: str | None = None) -> list[Detection]:
    result = model.predict(source=frame, conf=conf, imgsz=imgsz,
                           classes=list(class_map), device=device,
                           verbose=False, save=False)[0]
    detections = []
    if result.boxes is None:
        return detections
    boxes = result.boxes.cpu()
    height, width = frame.shape[:2]
    for xyxy, score, class_id in zip(boxes.xyxy.numpy(), boxes.conf.numpy(), boxes.cls.numpy()):
        label = class_map.get(int(class_id))
        if label is None or float(score) < conf:
            continue
        x1, y1, x2, y2 = map(int, xyxy)
        x1, x2 = max(0, min(x1, width - 1)), max(0, min(x2, width - 1))
        y1, y2 = max(0, min(y1, height - 1)), max(0, min(y2, height - 1))
        if x2 > x1 and y2 > y1:
            detections.append(Detection(label, float(score), (x1, y1, x2, y2)))
    return detections


def annotate_frame(frame: np.ndarray, detections: list[Detection]) -> np.ndarray:
    output = frame.copy()
    height, width = output.shape[:2]
    for detection in detections:
        x1, y1, x2, y2 = detection.xyxy
        color = COLORS[detection.label]
        cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)
        text = f'{detection.label.capitalize()} {detection.confidence:.2f}'
        font, scale, thickness = cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1
        (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)
        left = max(0, min(x1, width - tw - 8))
        label_height = th + baseline + 8
        top = max(0, min(y1 - label_height if y1 >= label_height else y1, height - label_height))
        cv2.rectangle(output, (left, top),
                      (min(width - 1, left + tw + 8), min(height - 1, top + label_height)), color, -1)
        cv2.putText(output, text, (left + 4, top + th + 4), font,
                    scale, (0, 0, 0), thickness, cv2.LINE_AA)
    return output


def process_video(input_path: str, output_path: str, model, class_map,
                  conf: float, imgsz: int, device: str | None,
                  fps_override: float | None = None, overwrite: bool = True) -> int:
    source, destination = Path(input_path).expanduser().resolve(), Path(output_path).expanduser().resolve()
    if not source.is_file():
        raise FileNotFoundError(f'Input video does not exist: {source}')
    if source == destination:
        raise ValueError('Input and output paths must differ.')
    codecs = {'.mp4': 'mp4v', '.avi': 'MJPG'}
    if destination.suffix.lower() not in codecs:
        raise ValueError('Output extension must be .mp4 or .avi.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    capture = cv2.VideoCapture(str(source))
    writer = None
    count = 0
    try:
        if not capture.isOpened():
            raise RuntimeError(f'Cannot open input: {source}')
        fps = fps_override if fps_override is not None else capture.get(cv2.CAP_PROP_FPS)
        if not math.isfinite(fps) or fps <= 0:
            fps = 30.0  # Fallback safe default FPS
        expected = capture.get(cv2.CAP_PROP_FRAME_COUNT)
        ok, frame = capture.read()
        if not ok:
            raise RuntimeError('Input contains no decodable frames.')
        height, width = frame.shape[:2]
        out_width, out_height = width + width % 2, height + height % 2
        writer = cv2.VideoWriter(str(destination),
                                 cv2.VideoWriter_fourcc(*codecs[destination.suffix.lower()]),
                                 fps, (out_width, out_height))
        if not writer.isOpened():
            raise RuntimeError('Cannot initialize encoder; try an .avi output (MJPG).')
        while ok:
            if frame.shape[:2] != (height, width):
                raise RuntimeError('Video frame dimensions changed during decoding.')
            detections = detect_frame(model, frame, class_map, conf, imgsz, device)
            annotated = annotate_frame(frame, detections)
            annotated = cv2.copyMakeBorder(annotated, 0, out_height - height,
                                           0, out_width - width, cv2.BORDER_CONSTANT)
            writer.write(annotated)
            count += 1
            if count % 100 == 0:
                logging.info('Processed %d frames', count)
            ok, frame = capture.read()
        if math.isfinite(expected) and expected > 0 and count < round(expected):
            logging.warning('Decoded %d of approximately %.0f frames; check for a damaged input.', count, expected)
    finally:
        capture.release()
        if writer is not None:
            writer.release()
    logging.info('Saved %d frames to %s (no audio)', count, destination)
    return count


def main() -> None:
    # Clear sys.argv to prevent Jupyter kernel arguments from crashing argparse
    sys.argv = [sys.argv[0]]
    
    logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
    try:
        model, class_map = load_model(MODEL_PATH, ALLOW_MISSING)
        process_video(VIDEO_PATH, OUTPUT_PATH, model, class_map, CONF_THRESH,
                      640, None, None, True)
    except Exception as exc:
        logging.error('%s', exc)
        raise


if __name__ == '__main__':
    main()